In [ ]:
import scanpy as sc
import pandas as pd
import rapids_singlecell as rsc
sc.set_figure_params(figsize=(3,3),dpi=150)

adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')

adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')

celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()

celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()

adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)

# Select epithelial cells supported by the global score ranking
mask = (adata_qc.obs['leiden_coarse'] == 'Epithelial Cells') & (adata_qc.obs['best_rank_type_global'] == 'Epi')

adata = adata_qc[mask].copy()

rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)

rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=20)

rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=10, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)

rsc.tl.leiden(adata, resolution=0.5, key_added='leiden_0.5_detailed')

sc.tl.rank_genes_groups(adata, groupby='leiden_0.5_detailed', method='t-test')

result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:  # iterate over each group
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/epi_deg_{group}_filtered_ttest.csv', index=False)


anno_dict = {
    '0': 'Epi_AQP2',      # Distal-nephron/collecting-duct principal cells (high AQP2 and DEFB1); most consistent with CNT/CCD principal-like epithelium with transitional TAL/DCT features.
    '1': 'Epi_CA12',       # Intercalated cells, biased toward an acid-secreting alpha-intercalated phenotype.
    '2': 'Epi_JUN',  # ccRCC tumor cells with a stress/high-metabolism program, biased toward ER stress and immediate-early responses.
    '3': 'Epi_CA9',       # Canonical core ccRCC tumor cells, most consistent with CA9-high hypoxic proliferative tumor-like epithelial cells.
    '4': 'Epi_ALDOB',      # Proximal-tubule cells biased toward organic-acid, amino-acid, and small-molecule catabolism.
    '5': 'Epi_CD44_high',       # T cells / NK cells
    '6': 'Epi_MIOX', # The most typical mature proximal-tubule-like population, biased toward a mature metabolic proximal-tubule state.
    '7': 'Epi_GPX3',     # Highly differentiated proximal-tubule epithelium (high BBOX1 and MIOX), with active antioxidant, detoxification, and respiratory-chain programs.
    '8': 'Epi_VIM'      # Proximal-tubule epithelium enriched for lipid metabolism and mitochondrial programs (high FABP1 and GATM), biased toward an EMT-like, hypoxic, inflammatory state.
}
adata.obs['cell_subtype'] = adata.obs['leiden_0.5_detailed'].map(anno_dict)

sc.pl.umap(adata, color='cell_subtype', title='Epi',save='_subtype_epi')

adata.write_h5ad('adata_epi.h5ad')





In [ ]:
adata = sc.read_h5ad('../adata_epi.h5ad')
adata

In [ ]:
adata.obsm["X_umap_backup"] = adata.obsm["X_umap"].copy()

In [ ]:
adata = adata[~adata.obs["cell_subtype"].isin(["Epi_CD44_high", "Epi_AQP2", "Epi_CA12"]),:].copy()

In [ ]:
markers = [
      'ACLY', 'ADAM17', 'AKT1','ALDH1B1','ALDH1L1','BMI1','CA9','CD24','CD34','CD44',
      'CDKN1A','CDKN2A','CTBP2','CXCR4','DHX9','DOT1L','E2F2','EGF','EPAS1',
      'EZH2','FKBP5','FZD4','GJA1','GLI1','HIF1A','HOTAIR','ID4','IGF1R','IL22',
      'ITGA6','JAK2','KDR','KLF4','KLF5','KLF9','L1TD1','LGR5','LILRB2','LIN28A',
      'MCAM','MICU1','NANOG','NCAM1',
      'NES','NOTCH1','PHLDA2','PLAU','POU5F1','PROM1','PSMD10','PTEN','PTTG1','RAC1',
      'RECK','REG4','S100A4','SCGB2A1','SEMA3F','SIRT2','SMAD6','SOX2','SPDYA','STAT3',
      'TERT','TGFB1','TGFBR2','TP53','TP63','TWIST1','USP7','YAP1','YBX1',
  ]

raw_genes = set(adata.raw.var_names)
markers_in_raw = [g for g in markers if g in raw_genes]
markers_not_in_raw = [g for g in markers if g not in raw_genes]

print("in adata.raw:")
print(markers_in_raw)

print("\nnot in adata.raw:")
print(markers_not_in_raw)


In [ ]:
sc.tl.score_genes(adata, gene_list=markers, score_name='stemness_score',use_raw=True)

In [ ]:
sc.tl.paga(adata, groups='cell_subtype')
sc.tl.umap(adata, init_pos='paga')

In [ ]:
#adata.uns["iroot"] = np.flatnonzero(adata.obs["cell_subtype"] == "Epi_ALDOB")[0]
mask = adata.obs["cell_subtype"] == "Epi_MIOX"
root_name = adata.obs.loc[mask, "stemness_score"].idxmin()
root_idx = adata.obs_names.get_loc(root_name)
adata.uns["iroot"] = root_idx
sc.tl.diffmap(adata)
sc.tl.dpt(adata)


In [ ]:
adata

In [ ]:
adata

In [ ]:
pt_mean = (
      adata.obs.groupby("cell_subtype")["dpt_pseudotime"]
      .mean()
      .sort_values()
  )

print(pt_mean)

In [ ]:
adata.write_h5ad('adata_epi_dpt_rootmiox.h5ad')

# Redraw DPT root-from-MIOX figures from saved h5ad

Reads `adata_epi_dpt_rootmiox.h5ad` and redraws the existing DPT/stemness figures with new filenames, without overwriting the original outputs.


In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Keep text editable in PDF outputs.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "DejaVu Sans"

BASE_DIR = Path('/mnt/disk18t/lr_xcy/riku/kirc_205/leiden_detailed/dpt_root_from_miox')
H5AD_PATH = BASE_DIR / 'adata_epi_dpt_rootmiox.h5ad'
FIG_DIR = BASE_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = FIG_DIR
sc.settings.verbosity = 2
print('input:', H5AD_PATH)
print('figdir:', sc.settings.figdir)


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print('raw:', adata.raw is not None)
print('obs keys:', [k for k in ['cell_subtype', 'stemness_score', 'dpt_pseudotime'] if k in adata.obs])
print('obsm keys:', list(adata.obsm.keys()))
print('has paga:', 'paga' in adata.uns)


In [ ]:
sc.set_figure_params(figsize=(3.5, 3.5), dpi=100)
sc.pl.embedding(
    adata,
    basis='X_umap_backup',
    color='stemness_score',
    color_map='viridis',
    save='stemness_score_X_umap_backup_size0p8_fig3p5_text_redraw.pdf',
    vmin=0,
    size=0.8,
)
plt.close('all')


In [ ]:
sc.set_figure_params(figsize=(5, 4.5), dpi=100)
sc.pl.violin(
    adata,
    keys='stemness_score',
    groupby='cell_subtype',
    stripplot=False,
    rotation=90,
    show=False,
)

axes = plt.gcf().get_axes()
for ax in axes:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.grid(False)

plt.savefig(BASE_DIR / 'violinplot_of_stemness_score_text_redraw.pdf', bbox_inches='tight')
plt.close('all')


In [ ]:
sc.pl.paga(
    adata,
    color='cell_subtype',
    node_size_scale=5,
    save='dpt_text_redraw.pdf',
)
plt.close('all')


In [ ]:
sc.pl.umap(
    adata,
    color=['dpt_pseudotime', 'cell_subtype'],
    save='_epi_paga_text_redraw.pdf',
)
plt.close('all')


In [ ]:
sc.pl.embedding(
    adata,
    basis='X_umap_backup',
    color=['dpt_pseudotime', 'cell_subtype'],
    save='umap_dpt_leiden_text_redraw.pdf',
)
plt.close('all')


In [ ]:
sc.pl.diffmap(
    adata,
    color=['dpt_pseudotime', 'stemness_score', 'cell_subtype'],
    color_map='viridis',
    save='_epi_text_redraw.pdf',
    components=['2,4'],
)
plt.close('all')


In [ ]:
expected = [
    BASE_DIR / 'violinplot_of_stemness_score_text_redraw.pdf',
    FIG_DIR / 'dotplot_pro_genes_text_redraw.pdf',
    FIG_DIR / 'umapumap_dpt_text_redraw.pdf',
    FIG_DIR / 'pagadpt_text_redraw.pdf',
    FIG_DIR / 'umap_epi_paga_text_redraw.pdf',
    FIG_DIR / 'X_umap_backupumap_dpt_leiden_text_redraw.pdf',
    FIG_DIR / 'diffmap_epi_text_redraw.pdf',
]
for p in expected:
    print(p, p.exists(), p.stat().st_size if p.exists() else None)


In [ ]:
WORK = Path.cwd()
H5AD_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/spatial/ST_analysis/mendeley_nc9bc8dn4m.1/h5ad")
OUT_ROOT = WORK / "adata_expression"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_FILES = {
    "h46t": "H46_T.h5ad",
    "j38t": "J38_T.h5ad",
    "r29t": "R29_T.h5ad",
    "r51t": "R51_T.h5ad",
    "r114t": "R114_T.h5ad",
    "rcor": "R_cor.h5ad",
    "rmed": "R_med.h5ad",
    "s15t": "S15_T.h5ad",
    "x49t": "X49_T.h5ad",
    "x98t": "X98_T.h5ad",
    "y7t": "Y7_T.h5ad",
    "y12t": "Y12_T.h5ad",
    "y27t": "Y27_T.h5ad",
    "z43t": "Z43_T.h5ad",
}

missing_files = [H5AD_DIR / filename for filename in SAMPLE_FILES.values() if not (H5AD_DIR / filename).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing input h5ad files: {missing_files}")

GENES_OF_INTEREST = [
    "MALAT1", "NEAT1", "KLF6", "VMP1", "ZBTB20", "NNMT", "JUN", "FOSB",
    "EGR1", "JUNB", "DUSP1", "FOS", "YBX3", "B2M", "LINC01320", "SOD2",
    "ATF3", "HLA-B", "HLA-A", "VIM",
]

sc.settings._vector_friendly = True
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "DejaVu Sans",
})

In [ ]:
def normalize_sample(adata, target_sum=1e4):
    adata.var_names_make_unique()
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    else:
        adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    return adata


def read_sample(sample):
    path = H5AD_DIR / SAMPLE_FILES[sample]
    adata = sc.read_h5ad(path)
    return normalize_sample(adata)


adatas = {sample: read_sample(sample) for sample in SAMPLE_FILES}
adata_h46t = adatas["h46t"]
adata_j38t = adatas["j38t"]
adata_r29t = adatas["r29t"]
adata_r51t = adatas["r51t"]
adata_r114t = adatas["r114t"]
adata_rcor = adatas["rcor"]
adata_rmed = adatas["rmed"]
adata_s15t = adatas["s15t"]
adata_x49t = adatas["x49t"]
adata_x98t = adatas["x98t"]
adata_y7t = adatas["y7t"]
adata_y12t = adatas["y12t"]
adata_y27t = adatas["y27t"]
adata_z43t = adatas["z43t"]

In [ ]:
def find_gene(adata, gene):
    if gene in adata.var_names:
        return gene
    upper_map = {str(v).upper(): v for v in adata.var_names}
    return upper_map.get(gene.upper())


def save_embedding(fig, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path.with_suffix(".svg"), dpi=300, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".pdf"), dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_gene_maps(adatas, genes, size=56, vmax=None):
    for name, adata in adatas.items():
        subdir = OUT_ROOT / name
        subdir.mkdir(parents=True, exist_ok=True)
        for gene in genes:
            real_gene = find_gene(adata, gene)
            if real_gene is None:
                continue
            kwargs = dict(
                color=real_gene,
                show=False,
                vmin=0,
                basis="spatial",
                s=size,
                title=f"{name}_{real_gene}",
                alpha=1,
                add_outline=False,
            )
            if vmax is not None:
                kwargs["vmax"] = vmax
            sc.pl.embedding(adata, **kwargs)
            save_embedding(plt.gcf(), subdir / f"{name}_{real_gene}")


plot_gene_maps(adatas, GENES_OF_INTEREST, size=56)

In [ ]:
plot_gene_maps(
    {"y7t": adatas["y7t"], "y12t": adatas["y12t"]},
    GENES_OF_INTEREST,
    size=80,
    vmax=5,
)

In [ ]:
def plot_goi_score_maps(adatas, size=56, vmax=5):
    for name, adata in adatas.items():
        if "GOI_score" not in adata.obs.columns:
            print(f"[{name}] skip GOI_score plot: GOI_score not found")
            continue
        subdir = OUT_ROOT / name
        subdir.mkdir(parents=True, exist_ok=True)
        sc.pl.embedding(
            adata,
            color="GOI_score",
            show=False,
            vmin=0,
            basis="spatial",
            title=f"{name}_GOI_score",
            s=size,
            vmax=vmax,
        )
        save_embedding(plt.gcf(), subdir / f"{name}_GOI_score")


plot_goi_score_maps(adatas, size=56, vmax=5)

In [ ]:
plot_goi_score_maps(
    {"y7t": adatas["y7t"], "y12t": adatas["y12t"]},
    size=80,
    vmax=5,
)